In [2]:
!pip install langchain langchain-community pypdf sentence-transformers chromadb langchain-huggingface
import warnings
warnings.filterwarnings('ignore')

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Loading the Document

In [16]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("/content/drive/MyDrive/crime-and-punishment.pdf")
pages = loader.load()

print(f"Number of Pages : {len(pages)}")

Number of Pages : 767


In [26]:
print(pages[0])

page_content='Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': '/content/drive/MyDrive/crime-and-punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}


In [33]:
print(f"Pages 0 : {pages[0].page_content[:500]}")

Pages 0 : Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky


#Split the Documents into Chunking

- chunk_size=500 → each chunk will be roughly 500 characters long
- chunk_overlap=50 → each chunk shares the last 50 characters with the next one, so sentences that get cut off at a boundary still have some context carried over
- RecursiveCharacterTextSplitter is smart about where it splits — it tries paragraph breaks first, then sentences, then words, rather than just chopping at a hard character count mid-word

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(pages)
print(f"Number of Chunks : {len(chunks)}")
print("----------Sample Chunk----------")
print(chunks[10].page_content)

Number of Chunks : 2856
----------Sample Chunk----------
received with extraordinary demonstrations of love and 
honour.
A few months later Dostoevsky died. He was followed to 
the grave by a vast multitude of mourners, who ‘gave the 
hapless man the funeral of a king.’ He is still probably the 
most widely read writer in Russia.
In the words of a Russian critic, who seeks to explain the 
feeling inspired by Dostoevsky: ‘He was one of ourselves, a 
man of our blood and our bone, but one who has suffered


- Tokenization — the sentence is broken into tokens (words/sub-words), each mapped to an integer ID.
- Transformer processing — those token IDs pass through the neural network, where self-attention updates each token's representation based on surrounding context.
- Pooling — the per-token vectors are combined (usually by averaging) into one single vector representing the whole sentence.
- Output — a fixed-size numerical vector (384 numbers for this model) that captures the sentence's meaning, ready for similarity comparisons.

# Load your embedding model

In [37]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# quick sanity check
test_vector = embedding_model.embed_query("Raskolnikov committed a murder")
print(f"Vector length: {len(test_vector)}")
print(test_vector[:10])  # just peek at first 10 numbers

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector length: 384
[-0.02266831509768963, 0.05258525162935257, -0.11964306235313416, 0.028071725741028786, 0.07252128422260284, -0.058730851858854294, 0.05589362233877182, 0.08325468003749847, 0.012849918566644192, 0.04283260181546211]


# Index your chunks into Chroma (your first vector store)

In [41]:
!pip install langchain_chroma

In [50]:
from langchain_chroma import Chroma

# delete old collection if it exists
try:
    vector_store.delete_collection()
except:
    pass

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="crime_and_punishment"
)
print("Indexing Complete....")
print("Number of Vectors : ", vector_store._collection.count())

Indexing Complete....
Number of Vectors :  2856


In [51]:
query = "Why did Raskolnikov decide to kill the pawnbroker?"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results) :
  print(f"-----Result{i+1}--------")
  print(doc.page_content)
  print()

-----Result1--------
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive question as to what motive impelled him

-----Result2--------
to express his joy fully, but he was in a fever of excitement as 
though a ton-weight had fallen off his heart. Now he had the 
right to devote his life to them, to serve them…. Anything 
might happen now! But he felt afraid to think of further 
possibilities and dared not let his imagination range. But 
Raskolnikov sat still in the same place, almost sullen and 
indifferent. Though he had been the most insistent on get -
ting rid of Luzhin, he seemed now the least concerned at

-----Result3--------
off, as though he had plenty of time left for consideration.
Again the same rubbish, the same eggshells lying about 
on the spiral stairs, again the open doors of the 

In [54]:
queries = [
    "Why did Raskolnikov decide to kill the pawnbroker?",
    "What is Sonia's role in the story?",
    "Describe the setting of St. Petersburg in the novel."
]

for q in queries:
    print(f"===== QUERY: {q} =====")
    results = vector_store.similarity_search(q, k=3)
    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(doc.page_content[:300])
        print()

===== QUERY: Why did Raskolnikov decide to kill the pawnbroker? =====
--- Result 1 ---
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive question as to what motive impelled him

--- Result 2 ---
to express his joy fully, but he was in a fever of excitement as 
though a ton-weight had fallen off his heart. Now he had the 
right to devote his life to them, to serve them…. Anything 
might happen now! But he felt afraid to think of further 
possibilities and dared not let his imagination range.

--- Result 3 ---
off, as though he had plenty of time left for consideration.
Again the same rubbish, the same eggshells lying about 
on the spiral stairs, again the open doors of the flats, again 
the same kitchens and the same fumes and stench coming 
from them. Raskolnikov had not been here since that da

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

text_splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks_large = text_splitter_large.split_documents(pages)
print(f"Number of large chunks: {len(chunks_large)}")

vectorstore_large = Chroma.from_documents(
    documents=chunks_large,
    embedding=embedding_model,
    collection_name="crime_and_punishment_large"
)
print(vectorstore_large._collection.count())

Number of large chunks: 1501
1501


In [57]:
queries = [
    "Why did Raskolnikov decide to kill the pawnbroker?",
    "What is Sonia's role in the story?",
    "Describe the setting of St. Petersburg in the novel."
]

for q in queries:
    print(f"===== QUERY: {q} =====")
    results = vectorstore_large.similarity_search(q, k=3)
    for i, doc in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(doc.page_content[:300])
        print()

===== QUERY: Why did Raskolnikov decide to kill the pawnbroker? =====
--- Result 1 ---
Crime and Punishment
had followed him then on his painful way! Raskolnikov at 
that moment felt and knew once for all that Sonia was with 
him for ever and would follow him to the ends of the earth, 
wherever fate might take him. It wrung his heart … but he 
was just reaching the fatal place.
He 

--- Result 2 ---
cast eyes. It was loathsome and unbearable for him to look. 
But in the end there was much that surprised him and he 
began, as it were involuntarily, to notice much that he had 
not suspected before. What surprised him most of all was 
the terrible impossible gulf that lay between him and all the 


--- Result 3 ---
101Free eBooks at Planet eBook.com
‘But I think, if you would not do it yourself, there’s no 
justice about it…. Let us have another game.’
Raskolnikov was violently agitated. Of course, it was all 
ordinary youthful talk and thought, such as he had often 
heard before in di

# Install Milvus Lite

# Index into Milvus (reuses your existing chunks_large + embedding_model)

In [59]:
!pip install langchain_milvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00


In [60]:
from langchain_milvus import Milvus

vectorstore_milvus = Milvus.from_documents(
    documents=chunks_large,
    embedding=embedding_model,
    connection_args={"uri": "./milvus_demo.db"},  # local file-based, no signup needed
    collection_name="crime_and_punishment_milvus"
)

print("Milvus indexing complete!")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Milvus indexing complete!


# Second Embedding Model

In [61]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_bge = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# rebuild chunks_large into a new Chroma collection using this new embedding model
from langchain_chroma import Chroma

vectorstore_bge = Chroma.from_documents(
    documents=chunks_large,
    embedding=embedding_model_bge,
    collection_name="crime_and_punishment_bge"
)

print("BGE indexing complete!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE indexing complete!


# Run all 3 queries across every configuration and compare

In [62]:
queries = [
    "Why did Raskolnikov decide to kill the pawnbroker?",
    "What is Sonia's role in the story?",
    "Describe the setting of St. Petersburg in the novel."
]

configs = {
    "Chroma + MiniLM + chunk500": vector_store,
    "Chroma + MiniLM + chunk1000": vectorstore_large,
    "Milvus + MiniLM + chunk1000": vectorstore_milvus,
    "Chroma + BGE + chunk1000": vectorstore_bge,
}

for config_name, store in configs.items():
    print(f"\n\n############ CONFIG: {config_name} ############")
    for q in queries:
        print(f"\n===== QUERY: {q} =====")
        results = store.similarity_search(q, k=3)
        for i, doc in enumerate(results):
            print(f"--- Result {i+1} ---")
            print(doc.page_content[:250])



############ CONFIG: Chroma + MiniLM + chunk500 ############

===== QUERY: Why did Raskolnikov decide to kill the pawnbroker? =====
--- Result 1 ---
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those who maintained this 
opinion, the criminal scarcely attempted to defend himself. 
To the decisive quest
--- Result 2 ---
to express his joy fully, but he was in a fever of excitement as 
though a ton-weight had fallen off his heart. Now he had the 
right to devote his life to them, to serve them…. Anything 
might happen now! But he felt afraid to think of further 
poss
--- Result 3 ---
off, as though he had plenty of time left for consideration.
Again the same rubbish, the same eggshells lying about 
on the spiral stairs, again the open doors of the flats, again 
the same kitchens and the same fumes and stench coming 
from them. Ra

===== QUERY: What is Sonia's role in the story? =====
--- Res

In [63]:
findings = """
# Retrieval Experiment Findings — Crime and Punishment

## Configurations Tested
1. Chroma + MiniLM (all-MiniLM-L6-v2) + chunk_size=500, overlap=50
2. Chroma + MiniLM + chunk_size=1000, overlap=100
3. Milvus + MiniLM + chunk_size=1000, overlap=100
4. Chroma + BGE-small (bge-small-en-v1.5) + chunk_size=1000, overlap=100

## Observations

### Chunk size
Larger chunks (1000/100) outperformed smaller chunks (500/50) for abstract,
interpretive queries (e.g. "What is Sonia's role"), since they preserve more
surrounding narrative context per chunk. Smaller chunks were noisier for
these query types but comparable for concrete/descriptive queries
(e.g. Petersburg setting), where relevant keywords appear densely.

### Vector store (Chroma vs Milvus)
With identical chunks and embeddings, Chroma and Milvus returned IDENTICAL
top-3 results across every query. Vector store choice did not affect
retrieval relevance in this experiment — both compute similarity search
over the same underlying vectors using the same metric. Vector store choice
is more likely to matter for scalability, deployment, and infra features
(filtering, hybrid search, cloud hosting) than raw retrieval quality.

### Embedding model (MiniLM vs BGE-small)
BGE-small produced noticeably more precise, on-topic results than MiniLM
for interpretive queries. For "why did Raskolnikov kill the pawnbroker,"
BGE surfaced passages directly describing the murder planning. For
"Sonia's role," BGE's top result was Raskolnikov's direct confession to
Sonia -- arguably the most relevant passage in the entire book for that
query. MiniLM's results were relevant but less precise/more diffuse.
Embedding model choice had a bigger impact on retrieval quality than
vector store choice.

## Recommended Setup
Best combination: Chroma (or Milvus) + BGE-small-en-v1.5 + chunk_size=1000,
chunk_overlap=100

Reasoning: Chunk size of 1000 with overlap balanced context and precision
better than smaller chunks. Embedding model had the largest impact on
retrieval relevance out of all variables tested -- BGE-small consistently
surfaced more directly relevant passages for abstract/interpretive queries
than MiniLM. Vector store choice (Chroma vs Milvus) made no measurable
difference to relevance in this experiment, so the simpler local option
(Chroma) is sufficient for this use case.
"""

print(findings)

with open("findings.md", "w") as f:
    f.write(findings)

print("Saved to findings.md")


# Retrieval Experiment Findings — Crime and Punishment

## Configurations Tested
1. Chroma + MiniLM (all-MiniLM-L6-v2) + chunk_size=500, overlap=50
2. Chroma + MiniLM + chunk_size=1000, overlap=100
3. Milvus + MiniLM + chunk_size=1000, overlap=100
4. Chroma + BGE-small (bge-small-en-v1.5) + chunk_size=1000, overlap=100

## Observations

### Chunk size
Larger chunks (1000/100) outperformed smaller chunks (500/50) for abstract,
interpretive queries (e.g. "What is Sonia's role"), since they preserve more
surrounding narrative context per chunk. Smaller chunks were noisier for
these query types but comparable for concrete/descriptive queries
(e.g. Petersburg setting), where relevant keywords appear densely.

### Vector store (Chroma vs Milvus)
With identical chunks and embeddings, Chroma and Milvus returned IDENTICAL
top-3 results across every query. Vector store choice did not affect
retrieval relevance in this experiment — both compute similarity search
over the same underlying vectors 